# Auditoría de cobertura municipal
## DENUE 2020 — DENUE 2025 — CONAPO

### Objetivo

Comparar la cobertura geográfica de las tres bases municipales ya procesadas antes de construir indicadores derivados o integrar nuevas fuentes.

Se verificará:

- número de municipios por fuente;
- unicidad de `CVEGEO`;
- municipios comunes entre DENUE 2020, DENUE 2025 y CONAPO;
- claves presentes en una fuente y ausentes en otra;
- diferencias territoriales potencialmente relevantes para el periodo 2020–2025.

Esta auditoría no construye todavía la densidad comercial ni realiza regresiones.

In [1]:
##########2. Librerías y rutas

from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

archivo_denue_2020 = (
    PROCESSED_DIR
    / "denue_2020_municipal.csv"
)

archivo_denue_2025 = (
    PROCESSED_DIR
    / "denue_2025_municipal.csv"
)

archivo_conapo = (
    PROCESSED_DIR
    / "conapo_municipal.csv"
)

print("DENUE 2020:", archivo_denue_2020.exists())
print("DENUE 2025:", archivo_denue_2025.exists())
print("CONAPO:", archivo_conapo.exists())

DENUE 2020: True
DENUE 2025: True
CONAPO: True


3. Cargar las tres bases procesadas

### 1. Carga y validación inicial

Se cargan exclusivamente las bases municipales previamente procesadas.

La llave de integración será `CVEGEO`, conservada como texto de cinco posiciones para evitar la pérdida de ceros a la izquierda.

In [3]:
denue_2020 = pd.read_csv(
    archivo_denue_2020,
    dtype={"CVEGEO": "string"}
)

denue_2025 = pd.read_csv(
    archivo_denue_2025,
    dtype={"CVEGEO": "string"}
)

conapo = pd.read_csv(
    archivo_conapo,
    dtype={"CVEGEO": "string"}
)

print("DENUE 2020:", denue_2020.shape)
print("DENUE 2025:", denue_2025.shape)
print("CONAPO:", conapo.shape)

DENUE 2020: (2465, 4)
DENUE 2025: (2477, 4)
CONAPO: (2475, 6)


In [4]:
############4. Control básico de CVEGEO
resumen_fuentes = pd.DataFrame({
    "fuente": [
        "DENUE 2020",
        "DENUE 2025",
        "CONAPO"
    ],
    "registros": [
        len(denue_2020),
        len(denue_2025),
        len(conapo)
    ],
    "CVEGEO_unicos": [
        denue_2020["CVEGEO"].nunique(),
        denue_2025["CVEGEO"].nunique(),
        conapo["CVEGEO"].nunique()
    ],
    "CVEGEO_duplicados": [
        denue_2020["CVEGEO"].duplicated().sum(),
        denue_2025["CVEGEO"].duplicated().sum(),
        conapo["CVEGEO"].duplicated().sum()
    ],
    "CVEGEO_nulos": [
        denue_2020["CVEGEO"].isna().sum(),
        denue_2025["CVEGEO"].isna().sum(),
        conapo["CVEGEO"].isna().sum()
    ]
})

display(resumen_fuentes)

,fuente,registros,CVEGEO_unicos,CVEGEO_duplicados,CVEGEO_nulos
0,DENUE 2020,2465,2465,0,0
1,DENUE 2025,2477,2477,0,0
2,CONAPO,2475,2475,0,0


In [5]:
###5. Construir los conjuntos municipales
claves_denue_2020 = set(
    denue_2020["CVEGEO"]
)

claves_denue_2025 = set(
    denue_2025["CVEGEO"]
)

claves_conapo = set(
    conapo["CVEGEO"]
)

In [6]:
######  6. Comparaciones dos a dos

auditoria_pares = pd.DataFrame([
    {
        "comparacion": "DENUE 2020 vs CONAPO",
        "comunes": len(
            claves_denue_2020 & claves_conapo
        ),
        "solo_primera": len(
            claves_denue_2020 - claves_conapo
        ),
        "solo_segunda": len(
            claves_conapo - claves_denue_2020
        )
    },
    {
        "comparacion": "DENUE 2025 vs CONAPO",
        "comunes": len(
            claves_denue_2025 & claves_conapo
        ),
        "solo_primera": len(
            claves_denue_2025 - claves_conapo
        ),
        "solo_segunda": len(
            claves_conapo - claves_denue_2025
        )
    },
    {
        "comparacion": "DENUE 2020 vs DENUE 2025",
        "comunes": len(
            claves_denue_2020 & claves_denue_2025
        ),
        "solo_primera": len(
            claves_denue_2020 - claves_denue_2025
        ),
        "solo_segunda": len(
            claves_denue_2025 - claves_denue_2020
        )
    }
])

display(auditoria_pares)

,comparacion,comunes,solo_primera,solo_segunda
0,DENUE 2020 vs CONAPO,2465,0,10
1,DENUE 2025 vs CONAPO,2474,3,1
2,DENUE 2020 vs DENUE 2025,2465,0,12


In [7]:
######### 7. Identificar las claves DENUE 2020 que no están en CONAPO

solo_denue_2020_vs_conapo = (
    claves_denue_2020
    - claves_conapo
)

detalle_solo_denue_2020 = (
    denue_2020[
        denue_2020["CVEGEO"]
        .isin(solo_denue_2020_vs_conapo)
    ]
    [
        [
            "CVEGEO",
            "entidad",
            "municipio",
            "EST_RETAIL_2020"
        ]
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "DENUE 2020 sin correspondencia en CONAPO:",
    len(detalle_solo_denue_2020)
)

display(detalle_solo_denue_2020)

DENUE 2020 sin correspondencia en CONAPO: 0


,CVEGEO,entidad,municipio,EST_RETAIL_2020


In [8]:
###########8. DENUE 2025 que no aparece en CONAPO
solo_denue_2025_vs_conapo = (
    claves_denue_2025
    - claves_conapo
)

detalle_solo_denue_2025 = (
    denue_2025[
        denue_2025["CVEGEO"]
        .isin(solo_denue_2025_vs_conapo)
    ]
    [
        [
            "CVEGEO",
            "entidad",
            "municipio",
            "EST_RETAIL_2025"
        ]
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "DENUE 2025 sin correspondencia en CONAPO:",
    len(detalle_solo_denue_2025)
)

display(detalle_solo_denue_2025)

DENUE 2025 sin correspondencia en CONAPO: 3


,CVEGEO,entidad,municipio,EST_RETAIL_2025
0,24059,San Luis Potosí,Villa de Pozos SLP,2060
1,25019,Sinaloa,Eldorado,643
2,25020,Sinaloa,Juan José Ríos,602


In [9]:
###9. CONAPO que no aparece en DENUE 2020
solo_conapo_vs_denue_2020 = (
    claves_conapo
    - claves_denue_2020
)

detalle_conapo_sin_denue_2020 = (
    conapo[
        conapo["CVEGEO"]
        .isin(solo_conapo_vs_denue_2020)
    ]
    [
        [
            "CVEGEO",
            "entidad",
            "municipio",
            "POB_2020",
            "POB_2025"
        ]
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "CONAPO sin correspondencia en DENUE 2020:",
    len(detalle_conapo_sin_denue_2020)
)

display(detalle_conapo_sin_denue_2020)

CONAPO sin correspondencia en DENUE 2020: 10


,CVEGEO,entidad,municipio,POB_2020,POB_2025
0,02006,Baja California,San Quintín,120771,133471
1,02007,Baja California,San Felipe,20391,20689
2,04012,Campeche,Seybaplaya,15496,16095
3,04013,Campeche,Dzitbalché,16818,17270
4,07125,Chiapas,Honduras de la Sierra,11997,12446
5,12082,Guerrero,Las Vigas,10133,10136
6,12083,Guerrero,Ñuu Savi,11369,11563
7,12084,Guerrero,Santa Cruz del Rincón,7254,7060
8,12085,Guerrero,San Nicolás,7141,7026
9,17036,Morelos,Hueyapan,8012,8406


In [10]:
######10. CONAPO que no aparece en DENUE 2025
solo_conapo_vs_denue_2025 = (
    claves_conapo
    - claves_denue_2025
)

detalle_conapo_sin_denue_2025 = (
    conapo[
        conapo["CVEGEO"]
        .isin(solo_conapo_vs_denue_2025)
    ]
    [
        [
            "CVEGEO",
            "entidad",
            "municipio",
            "POB_2020",
            "POB_2025"
        ]
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "CONAPO sin correspondencia en DENUE 2025:",
    len(detalle_conapo_sin_denue_2025)
)

display(detalle_conapo_sin_denue_2025)

CONAPO sin correspondencia en DENUE 2025: 1


,CVEGEO,entidad,municipio,POB_2020,POB_2025
0,07125,Chiapas,Honduras de la Sierra,11997,12446


In [11]:
#####11. Construir una matriz general de presencia

universo_claves = sorted(
    claves_denue_2020
    | claves_denue_2025
    | claves_conapo
)

matriz_cobertura = pd.DataFrame({
    "CVEGEO": universo_claves
})

matriz_cobertura["DENUE_2020"] = (
    matriz_cobertura["CVEGEO"]
    .isin(claves_denue_2020)
)

matriz_cobertura["DENUE_2025"] = (
    matriz_cobertura["CVEGEO"]
    .isin(claves_denue_2025)
)

matriz_cobertura["CONAPO"] = (
    matriz_cobertura["CVEGEO"]
    .isin(claves_conapo)
)

display(matriz_cobertura.head())

,CVEGEO,DENUE_2020,DENUE_2025,CONAPO
0,01001,True,True,True
1,01002,True,True,True
2,01003,True,True,True
3,01004,True,True,True
4,01005,True,True,True


In [12]:
patrones_cobertura = (
    matriz_cobertura
    .groupby(
        [
            "DENUE_2020",
            "DENUE_2025",
            "CONAPO"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="municipios")
    .sort_values(
        "municipios",
        ascending=False
    )
)

display(patrones_cobertura)

,DENUE_2020,DENUE_2025,CONAPO,municipios
3,True,True,True,2465
2,False,True,True,9
1,False,True,False,3
0,False,False,True,1


## Conclusión de la auditoría de cobertura DENUE–CONAPO

La comparación de las bases municipales procesadas permitió identificar la cobertura geográfica común entre DENUE 2020, DENUE 2025 y CONAPO.

Principales resultados:

- DENUE 2020 contiene 2,465 claves municipales.
- DENUE 2025 contiene 2,477 claves municipales.
- CONAPO contiene 2,475 claves municipales.
- Las tres fuentes comparten 2,465 claves municipales.
- No existen claves presentes en DENUE 2020 que estén ausentes de CONAPO.
- Existen 9 claves presentes en DENUE 2025 y CONAPO que no aparecen en DENUE 2020.
- Existen 3 claves presentes en DENUE 2025 que no aparecen en CONAPO.
- Existe 1 clave presente en CONAPO que no aparece en las bases DENUE seleccionadas.
- No se imputaron artificialmente valores cero a las diferencias de cobertura.

Las 2,465 claves comunes constituyen una muestra geográfica candidata para el proyecto. Sin embargo, esta muestra todavía no se considera definitiva, ya que posteriormente se revisarán posibles modificaciones territoriales y la disponibilidad de información en Censo 2020, ILMM 2020 y CONEVAL 2020.

In [13]:
####Guardamos también la matriz de cobertura

archivo_auditoria = (
    PROCESSED_DIR
    / "auditoria_cobertura_denue_conapo.csv"
)

matriz_cobertura.to_csv(
    archivo_auditoria,
    index=False,
    encoding="utf-8-sig"
)

print("Auditoría guardada correctamente.")
print(archivo_auditoria)

Auditoría guardada correctamente.
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\auditoria_cobertura_denue_conapo.csv


In [14]:
muestra_candidata = (
    matriz_cobertura[
        matriz_cobertura[
            ["DENUE_2020", "DENUE_2025", "CONAPO"]
        ].all(axis=1)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Municipios en muestra candidata:",
    f"{len(muestra_candidata):,}"
)

display(muestra_candidata.head())

Municipios en muestra candidata: 2,465


,CVEGEO,DENUE_2020,DENUE_2025,CONAPO
0,01001,True,True,True
1,01002,True,True,True
2,01003,True,True,True
3,01004,True,True,True
4,01005,True,True,True


In [15]:
archivo_muestra_candidata = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo.csv"
)

muestra_candidata.to_csv(
    archivo_muestra_candidata,
    index=False,
    encoding="utf-8-sig"
)

print("Muestra candidata guardada correctamente.")

Muestra candidata guardada correctamente.
